## Quality of Care Report

This report displays a compact year-level summary of quality-of-care indicators and points to generated map outputs.

In [ ]:
ROOT_PATH <- "~/workspace"
source(file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care", "utils", "snt_dhis2_quality_of_care.r"))

snt_environment <- get_setup_variables(SNT_ROOT_PATH = ROOT_PATH, packages = c("jsonlite", "data.table", "arrow", "sf", "ggplot2", "glue", "reticulate", "RColorBrewer", "dplyr", "writexl", "knitr", "scales", "gridExtra"))
config_json <- load_snt_config(snt_environment$CONFIG_PATH, "SNT_config.json")
COUNTRY_CODE <- config_json$SNT_CONFIG$COUNTRY_CODE
DHIS2_FORMATTED_DATASET <- config_json$SNT_DATASET_IDENTIFIERS$DHIS2_DATASET_FORMATTED
PIPELINE_PATH <- file.path(ROOT_PATH, "pipelines", "snt_dhis2_quality_of_care")
DATA_PATH <- file.path(ROOT_PATH, "data", "dhis2", "quality_of_care")
REPORT_OUTPUTS_PATH <- file.path(PIPELINE_PATH, "reporting", "outputs")
FIGURES_PATH <- file.path(REPORT_OUTPUTS_PATH, "figures")
dir.create(DATA_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(REPORT_OUTPUTS_PATH, recursive = TRUE, showWarnings = FALSE)
dir.create(FIGURES_PATH, recursive = TRUE, showWarnings = FALSE)

In [ ]:
# Load latest district-year output and build summary
qoc_ctx <- load_latest_quality_of_care_output(DATA_PATH, COUNTRY_CODE)
qoc <- qoc_ctx$qoc
latest_file <- qoc_ctx$latest_file

summary_tbl <- build_quality_of_care_summary(qoc)
summary_paths <- save_quality_of_care_summary_outputs(
  summary_tbl = summary_tbl,
  report_outputs_path = REPORT_OUTPUTS_PATH,
  country_code = COUNTRY_CODE
)

knitr::kable(summary_tbl, caption = "Quality of Care - Year-level summary")

cat(glue::glue("\nLoaded file: {latest_file}\n"))
cat(glue::glue("Map outputs folder: {FIGURES_PATH}\n"))
cat(glue::glue("Summary data saved to: {summary_paths$summary_parquet}, {summary_paths$summary_csv}, {summary_paths$summary_xlsx}\n"))

## Graphs by Year

In [ ]:
# Build and save year-level indicator charts
charts_file <- save_quality_of_care_summary_charts(
  summary_tbl = summary_tbl,
  figures_path = FIGURES_PATH,
  country_code = COUNTRY_CODE
)

if (!is.null(charts_file)) {
  cat(glue::glue("Combined charts saved: {charts_file}\n"))
} else {
  cat("No chart produced (no indicator columns available).\n")
}

## Maps by District and Year

Maps are generated directly from the quality-of-care data and district shapes.

In [ ]:
# Load shapes and regenerate yearly maps through shared utils
shapes <- load_country_file_from_dataset(
  dataset_id = DHIS2_FORMATTED_DATASET,
  country_code = COUNTRY_CODE,
  suffix = "_shapes.geojson",
  label = "DHIS2 shapes data"
)

save_quality_of_care_maps(
  qoc_dt = qoc,
  shapes_sf = shapes,
  figures_path = FIGURES_PATH
)

cat(glue::glue("Yearly maps saved in: {FIGURES_PATH}\n"))